# 0. mapping_fix.csv 생성 가이드

- 필요 파일: 1️⃣서울시 주요 120장소 목록.csv, 2️⃣police_region.csv, 3️⃣서울시 주요 120장소 영역.shx

- 생성될 파일: 1️⃣서울시_120개_geodata.csv, 2️⃣서울시_120개_hotspot.csv, 3️⃣mapping_fix.csv

- 순서
    - 1. 서울시 주요 120장소 목록의 NaN데이터 처리 및 확인
    - 2. 경찰청(전처리 전) 칼럼명 추출 및 확인
    - 3. 딕셔너리로 매핑 테이블 생성 및 정렬 후 저장
    - 4. 서울시 주요 120장소 영역의 Polygon에 따라 데이터 수정 (dataframe에서)
    - 5. 지도를 csv 로 저장 -> **서울시_120개_geodata.csv**
    - 6. 매핑 테이블 csv로 저장 -> **서울시_120개_hotspot.csv**




## ❗매핑 테이블 전처리 가이드
- 순서
    - 1. 서울시_120개_hotspot.csv으로 DataFrame 생성
    - 2. NO 칼럼의 Dtype 변경 (float -> int)
    - 3. 서울대공원, 덕수궁/광화문, 보신각의 AREA_GU 수정
        - 서울대공원: AREA_GU를 경기도 과천시로 수정
        - 덕수궁/광화문: 행을 2개로 늘려 서울 중구, 서울 종로구 생성 (AREA_GU이외 정보는 모두 동일)
        - 보신각: AREA_GU를 서울 종로구로 수정
    - 4. head, tail로 데이터 확인 -> 불필요한 0,1번 행 삭제
    - 5. 최종 매핑 테이블 csv 저장 -> **mapping_fix.csv**


# 1. 서울시 전처리

In [21]:
import pandas as pd

# 서울시 주요 120개 장소 목록과 police_region.csv의 gu 컬럼 결합해서 데이터 프레임 생성

# 데이터 불러오기 (인코딩 에러 대비 fallback 추가)
# 노트북이 project 폴더에 있으므로 상위 폴더의 data 디렉토리를 사용합니다.
# 정적 분석기 경고 방지용 초기화 (실제 값은 아래에서 덮어씀)
df_api = pd.DataFrame()
df_api = pd.read_csv("../data/서울시 주요 120장소 목록.csv", encoding="utf-8-sig")
try:
    df_region = pd.read_csv("../data/before/police_region.csv", encoding="utf-8-sig")
except UnicodeDecodeError:
    df_region = pd.read_csv("../data/before/police_region.csv", encoding="cp949")


In [22]:
df_api.columns


Index(['CATEGORY', 'NO', 'AREA_CD', 'AREA_NM', 'ENG_NM', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='object')

In [23]:
# unnamed: 0 컬럼 제거
for i in range(5,23):
    df_api = df_api.drop(columns=[f"Unnamed: {i}"])

In [24]:
df_api.tail(5)

,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
986,NaN,NaN,NaN,NaN,NaN
987,NaN,NaN,NaN,NaN,NaN
988,NaN,NaN,NaN,NaN,NaN
989,NaN,NaN,NaN,NaN,NaN
990,NaN,NaN,NaN,NaN,NaN


In [25]:
# 인덱스 120번 확인
df_api.iloc[119]

CATEGORY                  공원
NO                     120.0
AREA_CD               POI128
AREA_NM                 홍제폭포
ENG_NM      Hongje Waterfall
Name: 119, dtype: object

In [26]:
# 120번부터 990번까지 행 삭제
df_api = df_api.iloc[:120]

In [27]:
df_api

,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,관광특구,1.0,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
1,관광특구,2.0,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
2,관광특구,3.0,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...
3,관광특구,4.0,POI004,이태원 관광특구,Itaewon Special Tourist Zone
4,관광특구,5.0,POI005,잠실 관광특구,Jamsil Special Tourist Zone
...,...,...,...,...,...
115,공원,116.0,POI124,서대문독립공원,Seodaemun Independence Park
116,공원,117.0,POI125,안양천,Anyangcheon River
117,공원,118.0,POI126,여의서로,Yeouiseoro
118,공원,119.0,POI127,올림픽공원,Olympic Park


# 2. 경찰청 전처리

In [28]:
df_region.head()

,범죄대분류,범죄중분류,서울 종로구,서울 중구,서울 용산구,서울 성동구,서울 광진구,서울 동대문구,서울 중랑구,서울 성북구,...,서울 강서구,서울 구로구,서울 금천구,서울 영등포구,서울 동작구,서울 관악구,서울 서초구,서울 강남구,서울 송파구,서울 강동구
0,강력범죄,살인기수,0,3,5,2,1,2,3,1,...,3,2,2,2,1,1,4,3,4,0
1,강력범죄,살인미수등,1,2,5,3,1,5,2,1,...,2,5,4,8,0,3,6,13,1,3
2,강력범죄,강도,5,5,4,2,6,4,2,1,...,6,0,3,6,1,3,5,17,3,4
3,강력범죄,강간,31,21,52,22,49,25,35,32,...,53,30,31,58,20,88,52,161,71,24
4,강력범죄,유사강간,6,7,9,3,9,9,5,8,...,16,7,2,10,7,22,13,42,8,9


In [29]:
# 서울이 포함된 지역명 칼럼을 시리즈로
df_seoul=df_region.columns[2:]
df_seoul

Index(['서울 종로구', '서울 중구', '서울 용산구', '서울 성동구', '서울 광진구', '서울 동대문구', '서울 중랑구',
       '서울 성북구', '서울 강북구', '서울 도봉구', '서울 노원구', '서울 은평구', '서울 서대문구', '서울 마포구',
       '서울 양천구', '서울 강서구', '서울 구로구', '서울 금천구', '서울 영등포구', '서울 동작구', '서울 관악구',
       '서울 서초구', '서울 강남구', '서울 송파구', '서울 강동구'],
      dtype='object')

In [30]:
df_seoul=df_seoul.tolist()
df_seoul

['서울 종로구',
 '서울 중구',
 '서울 용산구',
 '서울 성동구',
 '서울 광진구',
 '서울 동대문구',
 '서울 중랑구',
 '서울 성북구',
 '서울 강북구',
 '서울 도봉구',
 '서울 노원구',
 '서울 은평구',
 '서울 서대문구',
 '서울 마포구',
 '서울 양천구',
 '서울 강서구',
 '서울 구로구',
 '서울 금천구',
 '서울 영등포구',
 '서울 동작구',
 '서울 관악구',
 '서울 서초구',
 '서울 강남구',
 '서울 송파구',
 '서울 강동구']

In [31]:
df_seoul=pd.DataFrame(df_seoul, columns=["AREA_GU"])
df_seoul

,AREA_GU
0,서울 종로구
1,서울 중구
2,서울 용산구
3,서울 성동구
4,서울 광진구
5,서울 동대문구
6,서울 중랑구
7,서울 성북구
8,서울 강북구
9,서울 도봉구


# 3. df_seoul과 df_api 조인

In [32]:
df_seoul.head()

,AREA_GU
0,서울 종로구
1,서울 중구
2,서울 용산구
3,서울 성동구
4,서울 광진구


In [33]:
df_api.head()

,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,관광특구,1.0,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
1,관광특구,2.0,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
2,관광특구,3.0,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...
3,관광특구,4.0,POI004,이태원 관광특구,Itaewon Special Tourist Zone
4,관광특구,5.0,POI005,잠실 관광특구,Jamsil Special Tourist Zone


In [34]:
import pandas as pd

# 1. 120개 장소에 대한 자치구 매핑 데이터 정의
mapping_dict = {
    # 관광특구 및 고궁
    '강남 MICE 관광특구': '서울 강남구', '동대문 관광특구': '서울 중구', '명동 관광특구': '서울 중구',
    '이태원 관광특구': '서울 용산구', '잠실 관광특구': '서울 송파구', '종로·청계 관광특구': '서울 종로구',
    '홍대 관광특구': '서울 마포구', '경복궁': '서울 종로구', '광화문·덕수궁': '서울 중구',
    '보신각': '서울 중구', '서울 암사동 유적': '서울 강동구', '창덕궁·종묘': '서울 종로구',

    # 주요 역세권
    '가산디지털단지역': '서울 금천구', '강남역': '서울 강남구', '건대입구역': '서울 광진구', '고덕역': '서울 강동구',
    '고속터미널역': '서울 서초구', '교대역': '서울 서초구', '구로디지털단지역': '서울 구로구', '구로역': '서울 구로구',
    '군자역': '서울 광진구', '대림역': '서울 영등포구', '동대문역': '서울 종로구', '뚝섬역': '서울 성동구',
    '미아사거리역': '서울 강북구', '발산역': '서울 강서구', '사당역': '서울 동작구', '삼각지역': '서울 용산구',
    '서울대입구역': '서울 관악구', '서울식물원·마곡나루역': '서울 강서구', '서울역': '서울 중구', '선릉역': '서울 강남구',
    '성신여대입구역': '서울 성북구', '수유역': '서울 강북구', '신논현역·논현역': '서울 강남구', '신도림역': '서울 구로구',
    '신림역': '서울 관악구', '신촌·이대역': '서울 서대문구', '양재역': '서울 서초구', '역삼역': '서울 강남구',
    '연신내역': '서울 은평구', '오목교역·목동운동장': '서울 양천구', '왕십리역': '서울 성동구', '용산역': '서울 용산구',
    '이태원역': '서울 용산구', '장지역': '서울 송파구', '장한평역': '서울 동대문구', '천호역': '서울 강동구',
    '총신대입구(이수)역': '서울 동작구', '충정로역': '서울 서대문구', '합정역': '서울 마포구', '혜화역': '서울 종로구',
    '홍대입구역(2호선)': '서울 마포구', '회기역': '서울 동대문구',

    # 발달상권 및 명소
    '가락시장': '서울 송파구', '가로수길': '서울 강남구', '광장(전통)시장': '서울 종로구', '김포공항': '서울 강서구',
    '노량진': '서울 동작구', '덕수궁길·정동길': '서울 중구', '북촌한옥마을': '서울 종로구', '서촌': '서울 종로구',
    '성수카페거리': '서울 성동구', '쌍문역': '서울 도봉구', '압구정로데오거리': '서울 강남구', '여의도': '서울 영등포구',
    '연남동': '서울 마포구', '영등포 타임스퀘어': '서울 영등포구', '용리단길': '서울 용산구', '이태원 앤틱가구거리': '서울 용산구',
    '인사동': '서울 종로구', '창동 신경제 중심지': '서울 도봉구', '청담동 명품거리': '서울 강남구', '청량리 제기동 일대 전통시장': '서울 동대문구',
    '해방촌·경리단길': '서울 용산구', 'DDP(동대문디자인플라자)': '서울 중구', 'DMC(디지털미디어시티)': '서울 마포구', '잠실롯데타워 일대': '서울 송파구',

    # 공원 및 한강지구
    '강서한강공원': '서울 강서구', '고척돔': '서울 구로구', '광나루한강공원': '서울 강동구', '광화문광장': '서울 종로구',
    '국립중앙박물관·용산가족공원': '서울 용산구', '난지한강공원': '서울 마포구', '남산공원': '서울 중구', '노들섬': '서울 용산구',
    '뚝섬한강공원': '서울 광진구', '망원한강공원': '서울 마포구', '반포한강공원': '서울 서초구', '북서울꿈의숲': '서울 강북구',
    '서리풀공원·몽마르뜨공원': '서울 서초구', '서울광장': '서울 중구', '서울대공원': '서울 과천시', # 행정구역상 과천이나 서울 관리
    '서울숲공원': '서울 성동구', '아차산': '서울 광진구', '양화한강공원': '서울 영등포구', '어린이대공원': '서울 광진구',
    '여의도한강공원': '서울 영등포구', '월드컵공원': '서울 마포구', '응봉산': '서울 성동구', '이촌한강공원': '서울 용산구',
    '잠실종합운동장': '서울 송파구', '잠실한강공원': '서울 송파구', '잠원한강공원': '서울 서초구', '청계산': '서울 서초구',
    '청와대': '서울 종로구', '보라매공원': '서울 동작구', '올림픽공원': '서울 송파구',

    # 테마 거리 및 주거지 인근
    '북창동 먹자골목': '서울 중구', '남대문시장': '서울 중구', '익선동': '서울 종로구', '신정네거리역': '서울 양천구',
    '잠실새내역': '서울 송파구', '잠실역': '서울 송파구', '송리단길·호수단길': '서울 송파구', '신촌 스타광장': '서울 서대문구',
    '서대문독립공원': '서울 서대문구', '안양천': '서울 양천구', '여의서로': '서울 영등포구', '홍제폭포': '서울 서대문구'
}

# 2. df_api에 AREA_GU 컬럼 생성 (매핑 적용)
# 데이터 내 공백이나 줄바꿈이 있을 수 있으므로 strip() 처리 권장
df_api['AREA_NM'] = df_api['AREA_NM'].str.strip()
df_api['AREA_GU'] = df_api['AREA_NM'].map(mapping_dict)

# 3. df_seoul과 df_api 병합
# df_seoul의 AREA_GU와 df_api의 AREA_GU를 기준으로 Join
df_merged = pd.merge(df_seoul, df_api, on='AREA_GU', how='left')

# 4. 결과 확인
print(f"결합 전 df_api 행 개수: {len(df_api)}")
print(f"결합 후 전체 행 개수: {len(df_merged)}")
display(df_merged.head(10))

결합 전 df_api 행 개수: 120
결합 후 전체 행 개수: 121


,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,서울 종로구,관광특구,6.0,POI006,종로·청계 관광특구,Jongno Cheonggye Special Toruist Zone
1,서울 종로구,고궁·문화유산,8.0,POI008,경복궁,Gyeongbokgung Palace
2,서울 종로구,고궁·문화유산,12.0,POI012,창덕궁·종묘,Changdeokgung Palace & Jongmyo Shrine
3,서울 종로구,인구밀집지역,23.0,POI024,동대문역,Dongdaemun station
4,서울 종로구,인구밀집지역,52.0,POI054,혜화역,Hyehwa station
5,서울 종로구,발달상권,57.0,POI060,광장(전통)시장,Gwangjang(Traditional) Market
6,서울 종로구,발달상권,61.0,POI066,북촌한옥마을,Bukchon Hanok Village
7,서울 종로구,발달상권,62.0,POI067,서촌,Seochon
8,서울 종로구,발달상권,71.0,POI078,인사동,Insa-dong
9,서울 종로구,공원,81.0,POI088,광화문광장,Gwanghwamun Square


In [35]:
df_merged=df_merged.sort_values(by='NO').reset_index(drop=True)

In [36]:
df_merged.head(10)

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,서울 강남구,관광특구,1.0,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
1,서울 중구,관광특구,2.0,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
2,서울 중구,관광특구,3.0,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...
3,서울 용산구,관광특구,4.0,POI004,이태원 관광특구,Itaewon Special Tourist Zone
4,서울 송파구,관광특구,5.0,POI005,잠실 관광특구,Jamsil Special Tourist Zone
5,서울 종로구,관광특구,6.0,POI006,종로·청계 관광특구,Jongno Cheonggye Special Toruist Zone
6,서울 마포구,관광특구,7.0,POI007,홍대 관광특구,HongDae Culture & Arts Special Tourist Zone
7,서울 종로구,고궁·문화유산,8.0,POI008,경복궁,Gyeongbokgung Palace
8,서울 중구,고궁·문화유산,9.0,POI009,광화문·덕수궁,Gwanghwamun & Deoksugung Palace
9,서울 중구,고궁·문화유산,10.0,POI010,보신각,Bosingak


In [37]:
df_merged.tail(10)

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
111,서울 송파구,발달상권,113.0,POI121,송리단길·호수단길,Songridan-gil·Hosudan-gil
112,서울 서대문구,발달상권,114.0,POI122,신촌 스타광장,Shinchon Star Plaza
113,서울 동작구,공원,115.0,POI123,보라매공원,Boramae Park
114,서울 서대문구,공원,116.0,POI124,서대문독립공원,Seodaemun Independence Park
115,서울 양천구,공원,117.0,POI125,안양천,Anyangcheon River
116,서울 영등포구,공원,118.0,POI126,여의서로,Yeouiseoro
117,서울 송파구,공원,119.0,POI127,올림픽공원,Olympic Park
118,서울 서대문구,공원,120.0,POI128,홍제폭포,Hongje Waterfall
119,서울 중랑구,NaN,NaN,NaN,NaN,NaN
120,서울 노원구,NaN,NaN,NaN,NaN,NaN


In [38]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   AREA_GU   121 non-null    object 
 1   CATEGORY  119 non-null    object 
 2   NO        119 non-null    float64
 3   AREA_CD   119 non-null    object 
 4   AREA_NM   119 non-null    object 
 5   ENG_NM    119 non-null    object 
dtypes: float64(1), object(5)
memory usage: 5.8+ KB


In [39]:
# import geopandas as gpd
# import matplotlib.pyplot as plt

# # 1. SHX 파일 불러오기
# gdf_area = gpd.read_file("../data/서울시 주요 120장소 영역.shp")

# # 2. 데이터 구조 및 좌표계(CRS) 확인

# print(gdf_area.crs)  # 좌표계 확인 (epsg:4326, 5179 등)
# print(gdf_area.head())


# # 3. 지도에 영역 그리기
# # 120개 장소의 영역(Polygon)이 지도 위에 시각화됩니다.
# gdf_area.plot(figsize=(10, 10), color='skyblue', edgecolor='black', alpha=0.6)
# plt.title("Seoul 120 Hotspots Spatial Areas")
# plt.show()

# # 4. (참고) df_api와 결합하여 자치구 정보 합치기
# # AREA_NM이 공통 컬럼이라면 아래와 같이 병합 가능합니다.
# # gdf_final = gdf_area.merge(df_api[['AREA_NM', 'AREA_GU']], on='AREA_NM', how='left')

In [40]:
# csv 파일로 저장
# gdf_area.to_csv("../data/서울시_120개_geodata.csv", index=False, encoding="utf-8-sig")

In [41]:
df_merged.to_csv("../data/서울시_120개_hotspot.csv", index=False, encoding="utf-8-sig")

# 4. df_merged 파일 불러와서 전처리

In [42]:
df = pd.read_csv("../data/서울시_120개_hotspot.csv", encoding="utf-8-sig")


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   AREA_GU   121 non-null    object 
 1   CATEGORY  119 non-null    object 
 2   NO        119 non-null    float64
 3   AREA_CD   119 non-null    object 
 4   AREA_NM   119 non-null    object 
 5   ENG_NM    119 non-null    object 
dtypes: float64(1), object(5)
memory usage: 5.8+ KB


In [44]:
# 서울대공원의 AREA_GU -> 경기도 과천시로 수정
# 광화문 및 덕수궁은 2개를 만들어서 하나느 서울 중구, 하나는 서울 종로구
# 보신각은 서울 종로구
df.head(10)

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,서울 강남구,관광특구,1.0,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
1,서울 중구,관광특구,2.0,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
2,서울 중구,관광특구,3.0,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...
3,서울 용산구,관광특구,4.0,POI004,이태원 관광특구,Itaewon Special Tourist Zone
4,서울 송파구,관광특구,5.0,POI005,잠실 관광특구,Jamsil Special Tourist Zone
5,서울 종로구,관광특구,6.0,POI006,종로·청계 관광특구,Jongno Cheonggye Special Toruist Zone
6,서울 마포구,관광특구,7.0,POI007,홍대 관광특구,HongDae Culture & Arts Special Tourist Zone
7,서울 종로구,고궁·문화유산,8.0,POI008,경복궁,Gyeongbokgung Palace
8,서울 중구,고궁·문화유산,9.0,POI009,광화문·덕수궁,Gwanghwamun & Deoksugung Palace
9,서울 중구,고궁·문화유산,10.0,POI010,보신각,Bosingak


In [45]:
print(df[df['NO'] == 92][['AREA_NM', 'AREA_GU']])
print(df[df['NO'] == 9][['AREA_NM', 'AREA_GU']])
print(df[df['NO'] == 10][['AREA_NM', 'AREA_GU']])
print(df[df['NO'] == 121][['AREA_NM', 'AREA_GU']])
print(df[df['NO'] == 122][['AREA_NM', 'AREA_GU']])

# print(df[df['NO'] == '92'][['AREA_NM', 'AREA_GU']])
# print(df[(df['AREA_GU'] == '서울 중구') & (df['NO'] == '9')][['AREA_NM', 'AREA_GU']])
# print(df[(df['AREA_GU'] == '서울 종로구') & (df['NO'] == '9')][['AREA_NM', 'AREA_GU']])

# print(df[df['NO'] == '121'][['AREA_NM', 'AREA_GU']])
# print(df[df['NO'] == '122'][['AREA_NM', 'AREA_GU']])


Empty DataFrame
Columns: [AREA_NM, AREA_GU]
Index: []
   AREA_NM AREA_GU
8  광화문·덕수궁   서울 중구
  AREA_NM AREA_GU
9     보신각   서울 중구
Empty DataFrame
Columns: [AREA_NM, AREA_GU]
Index: []
Empty DataFrame
Columns: [AREA_NM, AREA_GU]
Index: []


- 서울시_120개_hotspot에서
- 1. NO 는 int
- 2. 92번 경기도 과천시, 서울대공원 삽입 (그 외 정부는 서울시 주요 120장소 목록에 있음)
- 3. 9번 광화문덕수궁 -> 2개 행으로 하나는 서울 중구, 하나는 서울 종로구
- 4. 10번 보신각 -> 서울 종로구 변경
- 121,122 번 서울 중랑구, 서울 노원구 삽입


In [46]:
# 결측치 다 0으로 채우고 no dtype 변경
df = df.fillna(0)
df['NO'] = df['NO'].astype(int)


In [47]:
# NO dtype 확안
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   AREA_GU   121 non-null    object
 1   CATEGORY  121 non-null    object
 2   NO        121 non-null    int64 
 3   AREA_CD   121 non-null    object
 4   AREA_NM   121 non-null    object
 5   ENG_NM    121 non-null    object
dtypes: int64(1), object(5)
memory usage: 5.8+ KB


In [48]:
# df 칼럼 확인
df.columns

Index(['AREA_GU', 'CATEGORY', 'NO', 'AREA_CD', 'AREA_NM', 'ENG_NM'], dtype='object')

In [49]:
# 서울대공원 데이터 칼럼에 맞게 삽입 -> 공원,92,POI100,서울대공원,Seoul Grand Park
new_row = {'NO': 92, 'AREA_NM': '서울대공원', 'AREA_GU': '경기도 과천시', 'AREA_CD': 'POI100', 'ENG_NM': 'Seoul Grand Park', 'CATEGORY': '공원'}
# concat
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)



In [50]:
# 92번 확인
print(df[df['NO'] == 92][:])

     AREA_GU CATEGORY  NO AREA_CD AREA_NM            ENG_NM
121  경기도 과천시       공원  92  POI100   서울대공원  Seoul Grand Park


In [51]:
# 덕수궁 광화문
print(df[df['NO'] == 9][:])

  AREA_GU CATEGORY  NO AREA_CD  AREA_NM                           ENG_NM
8   서울 중구  고궁·문화유산   9  POI009  광화문·덕수궁  Gwanghwamun & Deoksugung Palace


In [52]:
# 서울 종로구인 덕수궁 광화문 행 하나 추가
new_row = {'NO': 9, 'AREA_NM': '광화문·덕수궁', 'AREA_GU': '서울 종로구', 'AREA_CD': 'POI009', 'ENG_NM': 'Gwanghwamun & Deoksugung Palace', 'CATEGORY': '고궁·문화유산'}
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [53]:
#확인
print(df[df['NO'] == 9][:])

    AREA_GU CATEGORY  NO AREA_CD  AREA_NM                           ENG_NM
8     서울 중구  고궁·문화유산   9  POI009  광화문·덕수궁  Gwanghwamun & Deoksugung Palace
122  서울 종로구  고궁·문화유산   9  POI009  광화문·덕수궁  Gwanghwamun & Deoksugung Palace


In [54]:
# NO=10 번의 AREA_GU -> 서울 종로구로 변경
df.loc[df['NO'] == 10, 'AREA_GU'] = '서울 종로구'


In [55]:
print(df[df['NO'] == 10][:])

  AREA_GU CATEGORY  NO AREA_CD AREA_NM    ENG_NM
9  서울 종로구  고궁·문화유산  10  POI010     보신각  Bosingak


In [56]:
# NO 기준 정렬 
df = df.sort_values(by='NO').reset_index(drop=True)

In [57]:
df.tail()

# 120개 장소에 9번 하나 추가 총 121


,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
118,서울 서대문구,공원,116,POI124,서대문독립공원,Seodaemun Independence Park
119,서울 양천구,공원,117,POI125,안양천,Anyangcheon River
120,서울 영등포구,공원,118,POI126,여의서로,Yeouiseoro
121,서울 송파구,공원,119,POI127,올림픽공원,Olympic Park
122,서울 서대문구,공원,120,POI128,홍제폭포,Hongje Waterfall


In [58]:
# 서울 중랑구, 노원구 데이터 추가
new_row_121 = {'NO': 121, 'AREA_NM': '서울 중랑구', 'AREA_GU': '서울 중랑구', 'AREA_CD': 'POI129', 'ENG_NM': 'Seoul Jungnang-gu', 'CATEGORY': '기타'}
new_row_122 = {'NO': 122, 'AREA_NM': '서울 노원구', 'AREA_GU': '서울 노원구', 'AREA_CD': 'POI130', 'ENG_NM': 'Seoul Nowon-gu', 'CATEGORY': '기타'}
df = pd.concat([df, pd.DataFrame([new_row_121, new_row_122])], ignore_index=True)

In [59]:
# 확인
print(df[df['NO'] == 121][:])
print(df[df['NO'] == 122][:])

    AREA_GU CATEGORY   NO AREA_CD AREA_NM             ENG_NM
123  서울 중랑구       기타  121  POI129  서울 중랑구  Seoul Jungnang-gu
    AREA_GU CATEGORY   NO AREA_CD AREA_NM          ENG_NM
124  서울 노원구       기타  122  POI130  서울 노원구  Seoul Nowon-gu


In [60]:
# 꼬리 확인
df.tail()

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
120,서울 영등포구,공원,118,POI126,여의서로,Yeouiseoro
121,서울 송파구,공원,119,POI127,올림픽공원,Olympic Park
122,서울 서대문구,공원,120,POI128,홍제폭포,Hongje Waterfall
123,서울 중랑구,기타,121,POI129,서울 중랑구,Seoul Jungnang-gu
124,서울 노원구,기타,122,POI130,서울 노원구,Seoul Nowon-gu


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   AREA_GU   125 non-null    object
 1   CATEGORY  125 non-null    object
 2   NO        125 non-null    int64 
 3   AREA_CD   125 non-null    object
 4   AREA_NM   125 non-null    object
 5   ENG_NM    125 non-null    object
dtypes: int64(1), object(5)
memory usage: 6.0+ KB


In [62]:
df.head()

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,서울 노원구,0,0,0,0,0
1,서울 중랑구,0,0,0,0,0
2,서울 강남구,관광특구,1,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
3,서울 중구,관광특구,2,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
4,서울 중구,관광특구,3,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...


In [63]:
# index0, 1 삭제
df = df.drop(index=[0, 1]).reset_index(drop=True)

In [64]:
df.head()

,AREA_GU,CATEGORY,NO,AREA_CD,AREA_NM,ENG_NM
0,서울 강남구,관광특구,1,POI001,강남 MICE 관광특구,Gangnam MICE Special Tourist Zone
1,서울 중구,관광특구,2,POI002,동대문 관광특구,Dongdaemun Fashion Town Special Tourist Zone
2,서울 중구,관광특구,3,POI003,명동 관광특구,Myeong-dong Namdaemun Bukchang-dong Da-dong Mu...
3,서울 용산구,관광특구,4,POI004,이태원 관광특구,Itaewon Special Tourist Zone
4,서울 송파구,관광특구,5,POI005,잠실 관광특구,Jamsil Special Tourist Zone


In [65]:
# AREA_MAPPING으로 csv 파일 저장
df.to_csv("../data/mapping_fix.csv", index=False, encoding="utf-8-sig")
